In [ ]:
IN_COLAB = 'google.colab' in str(get_ipython())
print(f'IN_COLAB={IN_COLAB}')


### Environment detection
We detect whether this notebook is running in Colab or locally so checkpoint and data paths resolve correctly.

### Configure paths and project imports
This mounts Drive on Colab, resolves project root, and adds it to `sys.path` for package imports.

In [ ]:
import sys
from pathlib import Path

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_PATH = Path('/content/drive/MyDrive/piano_model')
    BASE_PATH.mkdir(parents=True, exist_ok=True)
    PROJECT_ROOT = BASE_PATH / 'piano_midi_model'
else:
    cwd = Path.cwd().resolve()
    PROJECT_ROOT = cwd.parent if cwd.name == 'notebooks' else cwd
    BASE_PATH = PROJECT_ROOT

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f'PROJECT_ROOT={PROJECT_ROOT}')


### Install dependencies
This installs the correct requirement set for the current environment.

In [ ]:
import subprocess
import sys

req_file = PROJECT_ROOT / ('requirements_colab.txt' if IN_COLAB else 'requirements.txt')
if req_file.exists():
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(req_file)], check=False)
else:
    print(f'Requirements file not found: {req_file}')


### Load configs, tokenizer, and test loader
This prepares shared data artifacts used for generation and metric evaluation across all model families.

In [ ]:
from config import DataConfig, TrainConfig
from data.dataset import create_dataloaders
from data.tokenizer import PianoTokenizer

data_cfg = DataConfig()
train_cfg = TrainConfig(batch_size=8, grad_accumulation_steps=4, device='auto')

if IN_COLAB:
    data_cfg.maestro_path = str(BASE_PATH / 'maestro-v3.0.0')
    data_cfg.tokenizer_path = str(BASE_PATH / 'tokenizer.json')
    data_cfg.processed_path = str(BASE_PATH / 'processed')

tokenizer = PianoTokenizer.load(data_cfg.tokenizer_path)
_, _, test_loader = create_dataloaders(data_cfg, train_cfg)
print('Test batches:', len(test_loader))


### Load best checkpoint for each model family
We instantiate baseline, Mamba-only, and hybrid models and load their best checkpoints for side-by-side evaluation.

In [ ]:
from pathlib import Path

from safetensors.torch import load_file as safetensors_load_file

from config import ModelConfig
from model.baseline import PianoBaselineModel
from model.hybrid import PianoHybridModel

def _resolve_ckpt(path_candidates):
    for p in path_candidates:
        p = Path(p)
        if p.exists():
            return p
    return None

model_cfg = ModelConfig(vocab_size=data_cfg.vocab_size, max_sequence_length=data_cfg.max_sequence_length)

baseline = PianoBaselineModel(vocab_size=data_cfg.vocab_size, d_model=model_cfg.d_model, n_layers=2, dropout=model_cfg.dropout, max_sequence_length=data_cfg.max_sequence_length)
mamba_only = PianoHybridModel(ModelConfig(vocab_size=data_cfg.vocab_size, max_sequence_length=data_cfg.max_sequence_length, use_mamba=True, use_cfc=False))
hybrid = PianoHybridModel(ModelConfig(vocab_size=data_cfg.vocab_size, max_sequence_length=data_cfg.max_sequence_length, use_mamba=True, use_cfc=True))

ckpt_map = {
    'baseline': _resolve_ckpt([PROJECT_ROOT / 'checkpoints_baseline' / 'best_model.safetensors', BASE_PATH / 'checkpoints_baseline' / 'best_model.safetensors']),
    'mamba_only': _resolve_ckpt([PROJECT_ROOT / 'checkpoints_mamba_only' / 'best_model.safetensors', BASE_PATH / 'checkpoints_mamba_only' / 'best_model.safetensors']),
    'hybrid': _resolve_ckpt([PROJECT_ROOT / 'checkpoints_hybrid' / 'best_model.safetensors', BASE_PATH / 'checkpoints_hybrid' / 'best_model.safetensors']),
}

for name, p in ckpt_map.items():
    print(name, '->', p)

if ckpt_map['baseline'] is not None:
    baseline.load_state_dict(safetensors_load_file(str(ckpt_map['baseline']), device='cpu'))
if ckpt_map['mamba_only'] is not None:
    mamba_only.load_state_dict(safetensors_load_file(str(ckpt_map['mamba_only']), device='cpu'))
if ckpt_map['hybrid'] is not None:
    hybrid.load_state_dict(safetensors_load_file(str(ckpt_map['hybrid']), device='cpu'))

models = {'baseline': baseline, 'mamba_only': mamba_only, 'hybrid': hybrid}
device = 'cuda' if __import__('torch').cuda.is_available() else 'cpu'
for m in models.values():
    m.to(device)
    m.eval()
print('Loaded models to', device)


### Generate 5 continuations for 5 different test seeds per model
This builds a balanced qualitative+quantitative sample set for fair comparison across all model types.

In [ ]:
import random
from pathlib import Path

gen_root = BASE_PATH / 'generation_eval_outputs'
gen_root.mkdir(parents=True, exist_ok=True)

test_samples = []
for seed_batch, _ in test_loader:
    for i in range(seed_batch.shape[0]):
        test_samples.append(seed_batch[i].tolist())

random.shuffle(test_samples)
test_seeds = test_samples[:5]
print(f'Using {len(test_seeds)} test seeds.')

all_outputs = {}
for model_name, model in models.items():
    model_dir = gen_root / model_name
    model_dir.mkdir(parents=True, exist_ok=True)
    all_outputs[model_name] = []
    for seed_idx, seed_tokens in enumerate(test_seeds):
        seed_mid = model_dir / f'seed_{seed_idx}.mid'
        tokenizer.decode(seed_tokens, seed_mid)
        for sample_idx in range(5):
            gen_tokens = model.generate(seed_tokens=seed_tokens, max_new_tokens=data_cfg.continuation_length, temperature=0.9, top_p=0.95, top_k=50)
            cont_tokens = gen_tokens[len(seed_tokens):]
            gen_full_mid = model_dir / f'seed_{seed_idx}_sample_{sample_idx}_full.mid'
            gen_cont_mid = model_dir / f'seed_{seed_idx}_sample_{sample_idx}_continuation.mid'
            tokenizer.decode(gen_tokens, gen_full_mid)
            tokenizer.decode(cont_tokens, gen_cont_mid)
            all_outputs[model_name].append({'seed': seed_mid, 'generated_full': gen_full_mid, 'continuation': gen_cont_mid})

{k: len(v) for k, v in all_outputs.items()}


### Compute seed-vs-continuation metrics on all outputs
We aggregate consistency metrics across generated samples to compare how well each model preserves the seed's musical character.

In [ ]:
import numpy as np

from evaluation.metrics import compare_seed_vs_continuation, rhythmic_regularity

rows = []
for model_name, items in all_outputs.items():
    for item in items:
        m = compare_seed_vs_continuation(item['seed'], item['continuation'])
        m['model'] = model_name
        m['generated_rhythmic_regularity'] = rhythmic_regularity(item['continuation'])
        rows.append(m)

print(f'Total evaluated samples: {len(rows)}')
rows[:3]


### Display side-by-side piano rolls
These visual comparisons show temporal shape and note activity differences between the same seed and each model continuation.

In [ ]:
from utils.midi_utils import compare_pianorolls

for model_name, items in all_outputs.items():
    if not items:
        continue
    item = items[0]
    print(f'Pianoroll preview: {model_name}')
    compare_pianorolls(item['seed'], item['continuation'])


### Summary table by model
We report core metrics requested for this study: pitch consistency, note density ratio, and rhythmic regularity.

In [ ]:
from collections import defaultdict

metric_keys = ['pitch_class_cosine', 'note_density_ratio', 'generated_rhythmic_regularity']
agg = defaultdict(lambda: defaultdict(list))
for row in rows:
    model_name = row['model']
    for key in metric_keys:
        agg[model_name][key].append(float(row[key]))

summary = {}
for model_name, metrics in agg.items():
    summary[model_name] = {}
    for key in metric_keys:
        vals = np.asarray(metrics[key], dtype=np.float64)
        summary[model_name][f'{key}_mean'] = float(vals.mean()) if vals.size else float('nan')
        summary[model_name][f'{key}_std'] = float(vals.std()) if vals.size else float('nan')

for model_name, stats in summary.items():
    print('')
    print(model_name)
    for key, value in stats.items():
        print(f'  {key}: {value:.4f}')

summary


### Qualitative audio playback
For each model, we synthesize one sample to WAV (if `fluidsynth` is available) and embed it for listening-based comparison.

In [ ]:
import shutil
import subprocess

from IPython.display import Audio, display

soundfont_candidates = [
    '/usr/share/sounds/sf2/FluidR3_GM.sf2',
    '/usr/share/soundfonts/default.sf2',
]
soundfont = next((p for p in soundfont_candidates if Path(p).exists()), None)

if IN_COLAB and shutil.which('fluidsynth') is None:
    subprocess.run(['apt-get', '-y', 'install', 'fluidsynth'], check=False)

for model_name, items in all_outputs.items():
    for idx, item in enumerate(items):
        mid = item['generated_full']
        wav = Path(mid).with_suffix('.wav')
        if shutil.which('fluidsynth') and soundfont:
            cmd = ['fluidsynth', '-ni', soundfont, str(mid), '-F', str(wav), '-r', '44100']
            subprocess.run(cmd, check=False)
            if wav.exists():
                print(f'Audio preview: {model_name} sample {idx}')
                display(Audio(filename=str(wav), rate=44100))
        else:
            print(f'Skipping audio render for {model_name}; install fluidsynth + soundfont.')
            break


### Notes and observations
Use this section to record qualitative impressions, failure modes, and hypotheses for next experiments.

- Model you preferred subjectively:
- Observed rhythmic behavior:
- Harmonic/key consistency notes:
- Artifacts to fix in next run:
- Follow-up experiment ideas:
